# Notebook 00 — AMD Environment & Official YOLO26l
## Slide 28–32 · 让我们看清实验机器 / Know your lab machine

**What will you learn? / 本节学什么？**
- 这台机器的 CPU、GPU、ROCm、OpenVINO、Ultralytics 版本是什么 What CPU, GPU and software versions this machine really has
- 如何验证“官方 YOLO26l 预训练模型”的身份 How to verify the identity of the official pretrained YOLO26l
- 在 W7900 上跑一次真实检测 Run one real detection on the W7900

**One key idea / 一个核心原理**
> 在测量任何性能之前，必须先确认**模型跑在哪个设备上、模型到底是谁**。Before measuring anything, confirm **which device the model runs on and what the model actually is**.

In [ ]:
from pathlib import Path
import sys
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("repo:", REPO)

## 1. Run the baseline: inspect the machine / 第一步：看清机器
CPU 和 GPU 是两种完全不同的计算架构：CPU 核心少而强，擅长复杂控制流；GPU 核心成千上万，擅长大规模并行计算。
A CPU has few powerful cores for complex logic; a GPU has thousands of cores for massive parallel math.

In [ ]:
from course.setup_checks import cpu_info, rocm_info, full_report

cpu = cpu_info()
rocm = rocm_info()
print("CPU vendor     :", cpu["vendor"])
print("CPU model      :", cpu["model"])
print("Physical cores :", cpu["physical_cores"], " · threads:", cpu["threads"])
print("NUMA nodes     :", cpu["numa_nodes"])
print("Key ISA        :", cpu["flags_sample"])
print("GPU gfx arch   :", rocm["gfx"], " (gfx1100 = Radeon PRO W7900 class)")

In [ ]:
import torch, openvino as ov, ultralytics
print("PyTorch     :", torch.__version__, "| HIP:", torch.version.hip)
print("torch.cuda.is_available() :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name    :", torch.cuda.get_device_name(0))
print("OpenVINO    :", ov.__version__, "| devices:", ov.Core().available_devices)
print("Ultralytics :", ultralytics.__version__)

## 2. Load the official YOLO26l and verify its identity / 加载官方 YOLO26l 并验证身份
本课程**只允许**官方 Ultralytics `YOLO("yolo26l.pt")` COCO 检测权重（80 类）。所有六个实验共用同一份权重。This course uses **only** the official Ultralytics YOLO26l COCO detection weights (80 classes). All six labs share this single file.

In [ ]:
from course.model import load_official_yolo26l, identity_record

model = load_official_yolo26l()          # asserts 80 classes + detect task
ident = identity_record()
print("file    :", ident["file"])
print("size    :", f"{ident['size_bytes']/1e6:.1f} MB")
print("sha256  :", ident["sha256"])
print("classes :", ident["verified"]["num_classes"], "| task:", ident["verified"]["task"])
print("first 8 class names:", list(model.names.values())[:8])

## 3. One real inference on the W7900 / 在 W7900 上做一次真实推理
下面在 GPU 上对 COCO 验证子集的一张图片做检测，并画出边界框。
Run detection on one image of our fixed COCO validation subset and draw the bounding boxes.

In [ ]:
from course.benchmark import load_bench_images

images = load_bench_images()
print("benchmark subset:", len(images), "images, first:", images[0].name)

In [ ]:
import matplotlib.pyplot as plt

res = model.predict(source=str(images[0]), imgsz=640, device="cuda",
                    verbose=False)
r = res[0]
im = r.plot()                       # BGR image with boxes
n_det = len(r.boxes)
print(f"detections: {n_det}")
for c in r.boxes.cls[:10]:
    print(" -", model.names[int(c)])
plt.figure(figsize=(9, 6))
plt.imshow(im[:, :, ::-1]); plt.axis("off"); plt.title(f"YOLO26l on W7900 · {n_det} objects"); plt.show()

## 4. Try it yourself / 学生练习
**先预测再验证**：你觉得对大规模并行推理（同一张图片、同一模型），CPU 和 GPU 谁更快？快多少倍？把你的预测写在下面，然后到 Notebook 01 验证。

**Predict first, then verify**: which device will be faster for the same model and image — the AMD CPU or the W7900? Write your guess, then check it in Notebook 01.

然后修改 `STUDENT_IMAGE_INDEX`（0–119），换一张图片重新推理，观察检测类别变化。
Then change `STUDENT_IMAGE_INDEX` (0–119) to detect a different image.

In [ ]:
STUDENT_IMAGE_INDEX = 5        # <-- change me / 修改我 (0..119)

my_guess_cpu_vs_gpu = "GPU 大约快 ___ 倍"   # <-- write your guess / 写下预测
print("my guess:", my_guess_cpu_vs_gpu)

res = model.predict(source=str(images[STUDENT_IMAGE_INDEX]), imgsz=640,
                    device="cuda", verbose=False)
plt.figure(figsize=(9, 6))
plt.imshow(res[0].plot()[:, :, ::-1]); plt.axis("off")
plt.title(f"image #{STUDENT_IMAGE_INDEX} · {len(res[0].boxes)} objects"); plt.show()

**Takeaway / 一句话总结**
> 性能实验从“确认设备与模型身份”开始——CPU 型号、GPU 架构、运行时版本、权重 SHA256，一个都不能少。
> Every performance experiment starts with verified hardware and a > verified model identity — CPU model, GPU arch, runtime versions and > the weight SHA256.